# PVT and Monte Carlo Mismatch Characterization of a 130-nm CMOS Current-Starved Ring Oscillator Using Open-Source Tools

## Robustness Analysis of a CSRO-VCO in 130-nm CMOS Technology

First, this project presents the development and simulation-based characterization of a Current-Starved Ring Oscillator Voltage-Controlled Oscillator (CSRO-VCO) using SkyWater's 130-nm CMOS technology and open-source tools. The work focuses on the oscillator's behavior under process, supply-voltage and temperature (PVT) variations, as well as device mismatch effects.

Undoubtedly, the main goal is to evaluate the stability of the oscillation frequency and the tuning capability of the circuit with respect to the 2.4 GHz ISM band, with potential application in wireless communication systems, including Bluetooth.

For this reason, the project adopts a nominal reference frequency of 2.45 GHz, within the 2.400–2.4835 GHz range of the 2.4 GHz ISM band. The simulations investigate the influence of the control voltage, PVT conditions and statistical device variations on the oscillation frequency.


Therefore, the project aims to demonstrate a reproducible development flow for RF analog integrated circuits using open-source tools, documenting the simulation conditions, the results obtained and the limitations identified during characterization.

## Organization and Open-Source Tools

### Notebook organization

| Chapter | Main topic |
|---|---|
| **01** | Context |
| **02** | The CMOS Inverter |
| **03** | Ring Oscillator |
| **04** | Current-Starved Ring Oscillator-Based Voltage-Controlled Oscillator |
| **05** | PVT and Monte Carlo Mismatch |
| **06** | Reproducibility |
| **07** | Conclusions |

The sequence is:

$$
\boxed{
\text{Context}
\rightarrow
\text{Inverter}
\rightarrow
\text{Oscillator}
\rightarrow
\text{VCO}
\rightarrow
\text{PVT and MM Analysis}
\rightarrow
\text{Reproduction}
\rightarrow
\text{Conclusion}
}
$$


So far, the work is focused on the pre-layout characterization of the circuit, including tuning, PVT and mismatch analysis. Physical implementation, parasitic extraction and post-layout validation remain future steps, especially important to verify the impact of parasitic capacitances and interconnects on the oscillator's frequency and robustness.

### Open-source tools

| Tool | Purpose | Version | Maintainer |
|---|---|---|---|
| **SkyWater 130 nm PDK** | CMOS device models and design rules | sky130A | SkyWater Technology / Google |
| **Python** | Automation, data analysis and result processing | 3.10.12 | Python Software Foundation |
| **Xschem** | Schematic capture and editing | 3.4.8RC | Stefan Schippers |
| **ngspice** | Analog circuit simulation and SPICE analysis | 45 | ngspice Project |
| **Matplotlib** | Plot generation and visualization of simulation results | 3.10.7 | Matplotlib Project |
| **Git** | Version control, traceability and project reproducibility | 2.34.1 | Git Project |

## 1. Context

### What is the ISM band?

**ISM** stands for *Industrial, Scientific and Medical*. It identifies radio-frequency spectrum bands originally intended for industrial, scientific and medical applications. Some of these bands are also used by wireless communication systems.

Examples of the use of license-free bands:

 - **2.4 GHz band**: Wi-Fi, Bluetooth and Zigbee.
 - **5 GHz band**: Wi-Fi.
 - **868 MHz band**: Zigbee, LoRa or Sigfox.
 - **434 MHz band**
 - **RFID frequencies**: (13.56 MHz, 125 kHz, etc.).


The 2.4 GHz ISM band spans 2.400 GHz to 2.4835 GHz in most of the world. It is used by technologies such as Bluetooth, Wi-Fi and Zigbee. Since several systems share the same spectrum, interference may occur; therefore, protocols and equipment employ techniques such as channel division, frequency hopping and power control. Thus, the ISM designation does not mean unrestricted use: power limits, emissions and other requirements depend on each country's regulations.

<p align="center">
  <img src="figures/588_ISM_Bands.jpg" alt="Spectra" width="400">
</p>


### What is a VCO?

A **VCO** (*Voltage-Controlled Oscillator*) is an oscillator whose frequency can be adjusted by a control voltage, $Vin_{VCO}$. Its characteristic can be represented by a function $f(Vin_{VCO})$: by varying the voltage, the oscillation frequency changes within a tuning range.

<p align="center">
  <img src="figures/vco.jpg" alt="Voltage-controlled oscillator" width="400">
</p>

Thus, the idea of the whole project is to provide a VCO that operates covering the ISM band.

## 2. The CMOS Inverter

### What is a CMOS inverter?

The **CMOS inverter** is a one-bit logic gate that outputs the complement of the input signal: when the input is low, the output is high; when the input is high, the output is low. CMOS stands for *Complementary Metal-Oxide-Semiconductor* and describes the complementary use of PMOS and NMOS transistors.

<p align="center">
  <img src="figures/inv1.jpg" alt="inv" width="200">
</p>

### Structure and operation

The basic CMOS inverter has two transistors: a PMOS connected between the positive supply ($V_{DD}$) and the output, and an NMOS connected between the output and ground (GND). The gates of both transistors are connected to the input. The PMOS acts as the pull-up path, while the NMOS acts as the pull-down path.

<p align="center">
  <img src="figures/inversor.jpg" alt="inv" width="200">
</p>


| Input | PMOS | NMOS | Output |
|---|---|---|---|
| Low | On | Off | High |
| High | Off | On | Low |

When the input is low, the PMOS conducts and charges the output capacitance toward $V_{DD}$; the NMOS remains off. When the input is high, the NMOS conducts and discharges that capacitance toward ground; the PMOS is off. During the transition between levels, both may conduct for a short interval, creating a momentary current path between supply and ground.

<p align="center">
  <img src="figures/charge.jpg" alt="cg" width="200">
</p>

<p align="center">
  <img src="figures/discharge.jpg" alt="inv" width="200">
</p>

### Essential figures of merit

The **propagation delay** is the time between a transition at the inverter input and the corresponding transition at the output. Since rising and falling transitions follow different paths — the rising one is driven mainly by the PMOS and the falling one by the NMOS — their delays are measured separately.

- **$t_{PLH}$** (*Propagation Low-to-High*): delay associated with the output changing from low to high. In the inverter, this transition occurs when the input changes from high to low. It is measured as the interval between the input crossing 50% of $V_{DD}$ on its falling edge and the output crossing 50% of $V_{DD}$ on its rising edge.
- **$t_{PHL}$** (*Propagation High-to-Low*): delay associated with the output changing from high to low. In the inverter, this occurs when the input changes from low to high. It is measured as the interval between the input crossing 50% of $V_{DD}$ on its rising edge and the output crossing 50% of $V_{DD}$ on its falling edge.

<p align="center">
  <img src="figures/images.jpeg" alt="inv" width="400">
</p>

$$
t_p = \frac{t_{PLH} + t_{PHL}}{2}
$$

Finally, it is worth stressing the importance of this equation, since it will be fundamental for finding the operating frequency of our ring oscillator.


For a CMOS inverter, the delays can also be estimated from the transistors' electrical parameters and the load capacitance. Considering the first-order model:

$$
t_{PHL} = t_{HL} = R_n C_{ox} = \tau_n
$$

$$
t_{PLH} = t_{LH} = R_p C_{ox} = \tau_p = 3\tau_n
$$

where $R_n$ and $R_p$ represent the equivalent resistances of the NMOS and PMOS transistors, respectively, and $C_{ox}$ represents the load capacitance considered in the model. The relation $\tau_p = 3\tau_n$ follows from the assumption adopted for the equivalent resistances of the transistors.

Substituting these relations into the average propagation delay expression gives:

$$
t_p = \frac{t_{PLH}+t_{PHL}}{2}
=\frac{\tau_p+\tau_n}{2}
=\frac{3\tau_n+\tau_n}{2}
=2\tau_n
$$

## 3. Ring Oscillator

### What is a ring oscillator?


A **ring oscillator** is a feedback circuit formed by an odd number of inverter stages. In this section, a chain of **three CMOS inverters** is considered: the output of the third stage is connected to the input of the first. In a block representation, only the chain's input signal ($v_{in}$) and the output signal after the third stage ($v_{out}$) may be shown; the intermediate nodes exist physically but need not be exposed as measurement terminals.

<p align="center">
  <img src="figures/ro.jpg" alt="inv" width="400">
</p>


### Barkhausen criteria

The Barkhausen criteria establish two conditions for oscillation in a feedback system:
$$
|A(j\omega_0)\beta(j\omega_0)|\geq 1
$$

$$
\arg{A(j\omega_0)\beta(j\omega_0)}=2\pi k,\quad k\in\mathbb{Z}.
$$


The first criterion requires a loop gain equal to or greater than unity to start the oscillation, while the second requires a total phase equivalent to $360^\circ k$, ensuring constructive feedback. In steady state, the loop gain tends to unity.

<p align="center">
  <img src="figures/image.png" alt="dw" width="300">
</p>

In a ring oscillator with an odd number of inverters, the total inversion at low frequencies is $180^\circ$. Thus, the propagation delays introduce the additional phase shift needed to satisfy the phase criterion and sustain oscillation. These criteria are therefore necessary but not sufficient to guarantee oscillation.


### Period and frequency estimate

For three stages, a first-order estimate is:

$$
T_{osc} \approx \sum_{i=1}^{3}(t_{PHL,i}+t_{PLH,i}).
$$

If the three stages are approximately equal, i.e., $t_{PHL,i}\approx t_{PHL}$ and $t_{PLH,i}\approx t_{PLH}$, then:

$$
T_{osc} \approx 3(t_{PHL}+t_{PLH}).
$$

Using the average delay of one stage, $t_p=(t_{PHL}+t_{PLH})/2$, we obtain:

$$
T_{osc}\approx 6t_p,
\qquad
f_{osc}\approx \frac{1}{T_{osc}}
=\frac{1}{3(t_{PHL}+t_{PLH})}
=\frac{1}{6t_p}.
$$

## 4. Current-Starved Ring Oscillator – VCO

### Tuning by variable resistance

The frequency of a ring oscillator depends on the inverter delay. One way to adjust it is to control the resistance in the discharge path: by reducing $V_{cont}$, the control NMOS transistors conduct less, the output takes longer to fall, and the oscillation frequency decreases.

<p align="center">
  <img src="figures/csro.png" alt="inv" width="500">
</p>

This technique can make falling transitions much slower than rising ones, increasing asymmetry and phase noise. In addition, near the NMOS threshold voltage the resistance grows rapidly, making the tuning nonlinear; at very low values of $V_{cont}$, the circuit may stop oscillating. Finally, a fixed resistive path in parallel limits this increase and maintains a minimum frequency, but it does not eliminate the asymmetry and also reduces the maximum frequency.

In turn, the same can be done for the pull-up branch, so that the topology looks like this:

<p align="center">
  <img src="figures/csro-2.jpg" alt="inv" width="400">
</p>

### Theoretical estimate of the bias current

Before sizing the current-limiting transistors, the current required to reach the desired frequencies in the 2.4 GHz ISM band is estimated.

<p align="center">
  <img src="figures/modelo.jpg" alt="Ring oscillator model with current limiters" width="200">
</p>

Considering a three-stage CMOS ring oscillator, the oscillation frequency can be approximated by:

$$
f_0 = \frac{I}{NCV_{DD}}
$$

Solving for the current:

$$
I = f_0NCV_{DD}
$$

The following parameters are adopted:

- Capacitance per stage: $C = 3\,\text{fF}$;
- Number of stages: $N = 3$;
- Supply voltage: $V_{DD} = 1.8\,\text{V}$.

The average delay per stage is estimated by:

$$
t_{\text{center}} = \frac{1}{2Nf_0}
$$

#### Frequency of 2.400 GHz

The required current is:

$$
\begin{aligned}
I &= f_0NCV_{DD} \\
  &= (2.4\times10^9)(3)(3\times10^{-15})(1.8) \\
  &= 38.88\times10^{-6}\,\text{A}
\end{aligned}
$$

$$
\boxed{I = 38.88\,\mu\text{A}}
$$

The average delay per stage is:

$$
\begin{aligned}
t_{\text{center}} &= \frac{1}{2Nf_0} \\
&= \frac{1}{2(3)(2.4\times10^9)}
\end{aligned}
$$

$$
\boxed{t_{\text{center}} \approx 69.44\,\text{ps}}
$$

#### Center frequency of 2.450 GHz

The required current is:

$$
\begin{aligned}
I &= f_0NCV_{DD} \\
  &= (2.45\times10^9)(3)(3\times10^{-15})(1.8) \\
  &= 39.69\times10^{-6}\,\text{A}
\end{aligned}
$$

$$
\boxed{I = 39.69\,\mu\text{A}}
$$

The average delay per stage is:

$$
\begin{aligned}
t_{\text{center}} &= \frac{1}{2Nf_0} \\
&= \frac{1}{2(3)(2.45\times10^9)}
\end{aligned}
$$

$$
\boxed{t_{\text{center}} \approx 68.03\,\text{ps}}
$$

#### Frequency of 2.4835 GHz

The required current is:

$$
\begin{aligned}
I &= f_0NCV_{DD} \\
  &= (2.4835\times10^9)(3)(3\times10^{-15})(1.8) \\
  &= 40.2327\times10^{-6}\,\text{A}
\end{aligned}
$$

$$
\boxed{I \approx 40.23\,\mu\text{A}}
$$

The average delay per stage is:

$$
\begin{aligned}
t_{\text{center}} &= \frac{1}{2Nf_0} \\
&= \frac{1}{2(3)(2.4835\times10^9)}
\end{aligned}
$$

$$
\boxed{t_{\text{center}} \approx 67.11\,\text{ps}}
$$

#### Summary of results

| Frequency (GHz) | Current ($\mu$A) | Average delay per stage (ps) |
|:---:|---:|---:|
| 2.4000 | 38.88 | 69.44 |
| 2.4500 | 39.69 | 68.03 |
| 2.4835 | 40.23 | 67.11 |

The results provide an initial estimate of the current required to cover the 2.4000–2.4835 GHz range. The final sizing of the limiting transistors must be validated through simulation, taking into account parasitic capacitances and process, voltage and temperature (PVT) variations.

### Input control

The control configuration used so far has a limitation: two independent inputs are required to control the oscillation frequency, which makes a unified adjustment difficult.

To solve this problem, a current mirror is proposed, responsible for establishing a controlled bias current and distributing it to the ring oscillator stages. In this way, each stage receives a defined current, allowing the oscillation frequency to be controlled through a single reference input, simplifying the control interface and ensuring more uniform stage biasing.

Thus, the final circuit will look like this:

<p align="center">
  <img src="figures/csro-f.jpg" alt="Ring oscillator model with current limiters" width="500">
</p>

> 🛈 Note: The dimensional parameters and capacitance values shown are preliminary and were adopted arbitrarily for illustrative purposes. Their optimization should be carried out later through simulation.

### VCO simulation and characterization

Based on the theoretical current estimate presented above, the initial dimensional parameters of the transistors responsible for biasing and current limiting of the oscillator were defined.

From this configuration, simulations were performed to characterize the VCO's behavior and evaluate its ability to operate in the 2.4 GHz ISM band. The analysis was divided into three steps:

1. **Time-domain analysis of the output:** verification of the waveform generated by the oscillator, allowing evaluation of the oscillation stability and the behavior of the output signal.

2. **Tuning characteristic:** analysis of the oscillation frequency as a function of the control voltage ($V_{\text{cont}}$), allowing identification of the tuning range and verification of whether the VCO covers the desired frequency band.

3. **Tuning sensitivity ($K_{\text{VCO}}$):** evaluation of the frequency variation with respect to the control voltage, expressed by:

$$
K_{\text{VCO}}(V_{\text{cont}}) =
\frac{df_{\text{osc}}}{dV_{\text{cont}}}
$$

This quantity represents the oscillator's tuning gain, usually expressed in $\text{Hz/V}$ or $\text{MHz/V}$, and allows evaluation of the sensitivity of the frequency to variations of the control voltage.

#### VCO time response

<p align="center">
    <img src="figures/grafico_tensao_vs_tempo.png"
         alt="Time-domain waveform of the VCO output"
         width="700">
</p>

#### Tuning characteristic: frequency as a function of control voltage

<p align="center">
    <img src="figures/grafico_freq_vs_vcont.png"
         alt="Oscillation frequency as a function of control voltage"
         width="700">
</p>

#### Tuning gain: $K_{\text{VCO}}$ as a function of control voltage

<p align="center">
    <img src="figures/grafico_kvco_vs_vcont.png"
         alt="VCO tuning gain as a function of control voltage"
         width="700">
</p>

From these curves, it is possible to evaluate the effective tuning range, the linearity of the $f_{\text{osc}}(V_{\text{cont}})$ characteristic and the variation of the tuning gain across the control range. These results therefore make it possible to verify compliance with the frequency requirements established for the project.

## 5. PVT and Mismatch Characterization

### What is PVT?

PVT stands for *Process, Voltage and Temperature*, representing the three main sources of variation that affect the behavior of integrated circuits:

- **Process:** variations arising from fabrication steps, which affect transistor parameters such as threshold voltage ($V_{TH}$), carrier mobility and effective dimensions.
- **Voltage:** variations in the supply voltage ($V_{DD}$), which modify the biasing conditions and device performance.
- **Temperature:** changes in operating temperature, which influence carrier mobility, threshold voltage and transistor currents.

These variations can modify important circuit parameters, such as propagation delays, power consumption and oscillation frequency.

### What is Monte Carlo Mismatch?

*Monte Carlo Mismatch* analysis uses statistical simulations to evaluate the effects of random variations between devices fabricated on the same integrated circuit.

Unlike global process variations, *mismatch* represents local differences between transistors, even when they are subject to the same fabrication conditions. These differences can occur, for example, in the threshold voltage ($V_{TH}$), the saturation current and the effective dimensions of the devices.

In each simulation run, random variations are drawn for the transistor parameters, making it possible to evaluate the statistical dispersion of circuit performance.

### Why is this analysis important for the CSRO-VCO?

In the CSRO-VCO (*Current-Starved Ring Oscillator Voltage-Controlled Oscillator*), the oscillation frequency depends directly on the bias currents and on the propagation delays of the oscillator stages.

Consequently, process, voltage and temperature variations, together with *mismatch* between transistors, can cause significant shifts in the operating frequency range.

This behavior is particularly critical for the project, since the VCO must fully cover the 2.4 GHz ISM band, from 2.4000 to 2.4835 GHz. If the variations cause an excessive shift of the tuning range, the oscillator may fail to meet the required frequency limits, even if it performs adequately under nominal conditions.

### PVT with Mismatch: simulation setup

For this step, only TT (*Typical-Typical*) process conditions were considered. This choice rests on the premise that global process variations can be compensated by calibration or *trimming* of the devices.

**200 Monte Carlo Mismatch runs** were performed for each combination of supply voltage and temperature, keeping the TT process.

The parameters used were:

- **Process:** TT (*Typical-Typical*);
- **Monte Carlo runs:** 200 per operating condition;
- **Supply voltage:** from $1.71\,\text{V}$ to $1.89\,\text{V}$, in steps of $0.01\,\text{V}$;
- **Temperature:** from −25 °C to 85 °C, in steps of 5 °C.

In this way, different voltage and temperature combinations were evaluated, making it possible to observe both the shift of the tuning range and the frequency dispersion caused by *mismatch* between devices.

<p align="center">
    <img src="figures/histogram.png"
         alt="histograma"
         width="900">
</p>

The goal is to identify the most critical operating conditions and verify whether the CSRO-VCO tuning range remains within the limits required to cover the 2.4 GHz ISM band.

<p align="center">
    <img src="figures/bluetooth_metrics.png"
         alt=""
         width="700">
</p>


## 6. Reproducibility

Reproducing the results presented in this work requires prior installation and configuration of the tools used in the design, simulation and post-processing flow of the circuit.

The following dependencies are required:

- **SKY130 PDK:** set of models and design rules of the 130-nm CMOS technology, used for device characterization and circuit simulation.
- **Xschem:** schematic capture tool used to describe and build the circuits.
- **Ngspice:** SPICE simulator responsible for the electrical analysis and circuit simulations, including the PVT and Monte Carlo Mismatch characterizations.
- **Python:** environment used for data post-processing, metric extraction and plot generation for result analysis.

Besides installing these tools, it is necessary to ensure version compatibility, proper configuration of the PDK paths and availability of the device models used in the simulations.

With these dependencies properly configured, it becomes possible to reproduce the workflow, rerun the simulations and compare the results obtained with those presented in this study.

### Output signal

To simulate the VCO's operation, analyze the output voltage curve and save the data to .csv:

In [ ]:
from pathlib import Path
import subprocess
import sys

project_dir = Path.cwd()

scripts = [
    project_dir / "reproducibility" / "sinal" / "sinal.py",
    project_dir / "reproducibility" / "sint" / "sintonia.py",
]

for script in scripts:
    if not script.is_file():
        raise FileNotFoundError(f"Script not found: {script}")

    print(f"\nRunning {script.name}...")
    subprocess.run([sys.executable, str(script)], check=True)

print("\nBoth scripts finished successfully.")

### Tuning curve and VCO gain simulation

To simulate the device's operation (with PVT adjustable in the netlist), run the following code:

In [ ]:
from pathlib import Path
import subprocess
import sys

# Searches for the project root starting from the notebook's working directory.
project_dir = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "reproducibility" / "sint" / "sintonia.py").is_file()
    ),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Could not locate the reproducibility/sint folder.")

sint_dir = project_dir / "reproducibility" / "sint"



print("Running the tuning simulation...")
subprocess.run(
    [sys.executable, str(sint_dir / "sintonia.py")],
    cwd=sint_dir,
    check=True,
)

print("\nGenerating the tuning and Kvco plots...")
subprocess.run(
    [sys.executable, str(sint_dir / "sintoniapos.py")],
    cwd=sint_dir,
    check=True,
)

print("\nTuning flow completed.")

### PVT and mismatch simulation

In [ ]:
from pathlib import Path
import subprocess
import sys

scripts_dir = Path.cwd() / "reproducibility" / "pvtmm"

copynet = scripts_dir / "copynet.py"
pipeline = scripts_dir / "rodar_pipeline.py"

for script in (copynet, pipeline):
    if not script.is_file():
        raise FileNotFoundError(f"Script not found: {script}")

    print(f"\nRunning {script.name}...")
    subprocess.run([sys.executable, str(script)], cwd=scripts_dir, check=True)

print("\nExecution completed successfully.")

> 🛈 Warning: the simulations are heavy and may take from minutes to days. If you still cannot run them, execute the .py files manually from the terminal, inside the directory of the executed code.

> If you need to change the circuit, open the .sch file in Xschem and make the changes you want; when you click "generate netlist" (configure a SPICE netlist), it will generate a .spice netlist that can be run by this notebook's Python scripts.

## 7. Conclusion

The characterization carried out made it possible to evaluate the behavior of the CSRO-VCO under different operating conditions, considering variations in process, supply voltage, temperature and *mismatch* between devices.

The simulation results indicate that the circuit shows reproducible behavior under most of the conditions evaluated, provided the established operating limits are respected. However, it was observed that voltage and temperature variations, combined with *mismatch*, can cause significant shifts in the oscillation frequency range.

This behavior shows that the nominal transistor sizing and the definition of the bias current, by themselves, are not sufficient to guarantee coverage of the 2.4 GHz ISM band under all operating conditions.

Therefore, the robustness of the design depends on an adequate definition of the bias current, on the available tuning margin, and on the implementation of calibration mechanisms (*trimming*) capable of compensating the frequency deviations caused by device variations.

We conclude that PVT and Monte Carlo Mismatch analysis is indispensable to assess the feasibility of the CSRO-VCO, identify the critical operating conditions and establish the requirements for a robust implementation in 130-nm CMOS technology.

## 8. References

RAZAVI, Behzad. **Design of Analog CMOS Integrated Circuits**. 2nd ed. New York: McGraw-Hill, 2017.

RAZAVI, Behzad. **Design of CMOS Phase-Locked Loops: From Circuit Level to Architecture Level**. Cambridge: Cambridge University Press, 2020.

ALEXANDER, Charles K.; SADIKU, Matthew N. O. **Fundamentos de circuitos elétricos** [Fundamentals of Electric Circuits]. 5th ed. Porto Alegre: AMGH, 2013.

LAGRIDA. **Online LaTeX Equation Editor**. 2025. Available at: <https://latexeditor.lagrida.com/>. Accessed: Oct 30, 2026.

SEDRA, Adel S.; SMITH, Kenneth C. **Microelectronic Circuits**. 7th ed. New York: Oxford University Press, 2015.

STEER, Michael. **Microwave and RF Design: Networks**. 3rd ed. Raleigh: North Carolina State University, 2019.

WIEDEMANN, Philipp; KIRZINGER, Manuel; PFANNENMÜLLER, Christof. **CircuiTikZ Designer**. 2025. Available at: <https://circuit2tikz.tf.fau.de/designer/>. Accessed: Oct 30, 2026.

OPENAI. **ChatGPT**. 2026. Tool used as an aid in reviewing and improving the writing. Available at: <https://chatgpt.com/>. Accessed: Oct 30, 2026.